In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260615_154932"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))

snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,ask_delta,quote_churn,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1781516013114,BTCUSDT,65606.06,65606.07,65606.065,65606.064810,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65609.215,0.000048
1,1781516013214,BTCUSDT,65606.06,65606.07,65606.065,65606.064869,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65606.685,0.000009
2,1781516013314,BTCUSDT,65606.06,65606.07,65606.065,65606.064708,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65606.065,0.000000,65614.875,0.000134,65606.685,0.000009
3,1781516013414,BTCUSDT,65606.06,65606.07,65606.065,65606.064708,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65609.355,0.000050,65614.875,0.000134,65606.685,0.000009
4,1781516013514,BTCUSDT,65606.06,65606.07,65606.065,65606.064860,6560606,6560607,6560606,0.01,...,0.0,0.0,65606.065,0.0,65614.875,0.000134,65614.875,0.000134,65606.685,0.000009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225582,1781538571914,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225583,1781538572014,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225584,1781538572114,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN
225585,1781538572214,BTCUSDT,67161.98,67161.99,67161.985,67161.987625,6716198,6716199,6716198,0.01,...,0.0,0.0,67161.985,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
snapshots["spread_bps"] = (
    snapshots.spread /
    snapshots.mid *
    10000
)
snapshots.spread_bps.describe()

count    225587.000000
mean          0.001858
std           0.022537
min           0.001487
25%           0.001501
50%           0.001508
75%           0.001516
max           4.875403
Name: spread_bps, dtype: float64

In [4]:
snapshots["spread_ticks"] = (
    snapshots.best_ask_tick -
    snapshots.best_bid_tick
)

snapshots.spread_ticks.describe()

count    225587.000000
mean          1.231352
std          14.917428
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max        3235.000000
Name: spread_ticks, dtype: float64

In [3]:
snapshots[
    snapshots.spread_bps > 0.01
][
    [
        "ts",
        "spread",
        "spread_bps",
        "mid",
        "volatility",
        "quote_churn"
    ]
].head()

,ts,spread,spread_bps,mid,volatility,quote_churn
3344,1781516347525,0.32,0.048837,65523.850,0.000021,0.0
6492,1781516662314,0.24,0.036609,65557.890,0.000035,0.0
6790,1781516692122,0.47,0.071681,65568.225,0.000033,0.0
46045,1781520617715,0.82,0.124715,65750.020,0.000002,0.0
48674,1781520880623,0.69,0.104857,65803.655,0.000184,0.0


In [ ]:
adverse_selection_bps =
    adverse_move / fill_price * 10000

net_edge_bps = (
    captured_spread_bps
    - 4
    - adverse_selection_bps
)